In [1]:
import fitz

In [2]:
doc = fitz.open('../data/tata-motor.pdf')

print(f"The document has {doc.page_count} pages.")


The document has 590 pages.


# Finding the Headings: Just detect patterns, assign meaning, and build heirarchy

In [3]:
all_spans = []
for page_num,page in enumerate(doc):
    blocks = page.get_text("dict")["blocks"]

    for b in blocks:
        if "lines" not in b:
            continue
        for line in b["lines"]:
            for span in line["spans"]:
                text = span["text"].strip()

                if not text:
                    continue

                all_spans.append({
                    "text":text,
                    "size": span["size"],
                    "font": span["font"],
                    "flags": span["flags"],
                    "page": page_num,
                    "y": span["bbox"][1]
                })

## Understanding the docs statistically

In [4]:
sizes = [s["size"] for s in all_spans]

avg_size = sum(sizes) / len(sizes)
max_size = max(sizes)
min_size = min(sizes)

print(avg_size,max_size,min_size)

8.713787477229904 75.0 4.647579669952393


In [5]:
heading_thresh = 0.75
subheading_thresh = 1.2


In [6]:
## Heading are often bold
def is_bold(flags):
    return flags & 2 != 0


In [7]:
def classify(span, avg_size, max_size, heading_thresh=0.75,subheading_thresh=1.2):
    text = span["text"]
    size = span["size"]
    bold = is_bold(span["flags"])

    ## Ignore noisy tiny text
    if len(text) < 2:
        return "ignore"
    
    ## Main Heading
    if size > max_size*heading_thresh and bold:
        return "h1"
    
    ## Subheading
    elif size > avg_size * subheading_thresh:
        return "h2"
    
    # Body   
    else:
        return "body"
    
    

In [8]:
structure = []

current_h1 = None
current_h2 = None
i=0
for span in all_spans:
    if(i == 5):
        break
    label = classify(span, avg_size, max_size)
    text = span["text"]
    print(f"The text is : {text}, and it's label is {label}, and all details are {span}")
    i = i+1
    
    

The text is : 80, and it's label is h2, and all details are {'text': '80', 'size': 11.0, 'font': 'Calibri-Bold', 'flags': 16, 'page': 0, 'y': 89.02040100097656}
The text is : TH, and it's label is body, and all details are {'text': 'TH', 'size': 6.413000106811523, 'font': 'Calibri-Bold', 'flags': 17, 'page': 0, 'y': 90.12308502197266}
The text is : INTEGRATED ANNUAL, and it's label is h2, and all details are {'text': 'INTEGRATED ANNUAL', 'size': 11.0, 'font': 'Calibri-Bold', 'flags': 16, 'page': 0, 'y': 89.02040100097656}
The text is : REPORT 2024-25, and it's label is h2, and all details are {'text': 'REPORT 2024-25', 'size': 11.0, 'font': 'Calibri-Bold', 'flags': 16, 'page': 0, 'y': 101.02137756347656}
The text is : In a free enterprise, the community is not just, and it's label is h2, and all details are {'text': 'In a free enterprise, the community is not just', 'size': 15.0, 'font': 'Calibri-Bold', 'flags': 16, 'page': 1, 'y': 607.6649780273438}


In [9]:
structure

[]

## We should not have started with Span, this is very small, Let's try with line level

In [10]:
def build_lines(blocks, page_num):
    lines_out = []

    for b in blocks:
        if "lines" not in b:
            continue

        for line in b["lines"]:
            spans = line["spans"]

            text = " ".join([s["text"].strip() for s in spans if s["text"].strip()])

            if not text:
                continue

            font_sizes = [s["size"] for s in spans]
            avg_size = sum(font_sizes) / len(font_sizes)

            is_bold = any((s["flags"] & 2) != 0 for s in spans)

            lines_out.append({
                "text":text,
                "size": avg_size,
                "is_bold":is_bold,
                "page": page_num,
                "bbox": line["bbox"]
            })      
    return lines_out



#### Use percentile instead of hardcoded values for threshold

In [11]:
import numpy as np

all_lines = []
for page_num,page in enumerate(doc):
    blocks = page.get_text("dict")["blocks"]
    all_lines.extend(build_lines(blocks,page_num))


sizes = [l["size"] for l in all_lines]

p90 = np.percentile(sizes,90)
p75 = np.percentile(sizes,75)
p50 = np.percentile(sizes,50)

In [12]:
def is_noise(text):
    if len(text) < 3:
        return True
    
    if text.isdigit():
        return True
    
    if len(text.split()) == 1 and len(text) < 4:
        return True
    
    return False

In [13]:
def classify_lines(line, p90,p75):
    text = line["text"]

    if len(text) < 3:
        return "ignore"
    
    ## Header
    if line["size"] >= p90 and line["is_bold"]:
        return "h1"
    
    ## Sub
    elif line["size"] >= p75:
        return "h2"
    
    else:
        return "body"

In [14]:
current_h1 = None
current_h2 = None
i=0
for line in all_lines:
    if(i == 5):
        break

    if(is_noise(line["text"])):
        continue
    
    label = classify_lines(line, p90, p75)
    text = line["text"]
    print(f"The text is : {text}, and it's label is {label}, and all details are {line}")
    i = i+1

The text is : 80 TH INTEGRATED ANNUAL, and it's label is body, and all details are {'text': '80 TH INTEGRATED ANNUAL', 'size': 9.471000035603842, 'is_bold': False, 'page': 0, 'bbox': (49.827701568603516, 89.02040100097656, 172.60302734375, 104.28839874267578)}
The text is : REPORT 2024-25, and it's label is h2, and all details are {'text': 'REPORT 2024-25', 'size': 11.0, 'is_bold': False, 'page': 0, 'bbox': (49.82870101928711, 101.02137756347656, 125.52302551269531, 116.28937530517578)}
The text is : In a free enterprise, the community is not just, and it's label is h2, and all details are {'text': 'In a free enterprise, the community is not just', 'size': 15.0, 'is_bold': False, 'page': 1, 'bbox': (155.38999938964844, 607.6649780273438, 443.2835998535156, 628.4849853515625)}
The text is : another stakeholder in business, but is in fact, and it's label is h2, and all details are {'text': 'another stakeholder in business, but is in fact', 'size': 15.0, 'is_bold': False, 'page': 1, 'bbox

In [15]:
import pandas as pd

df = pd.DataFrame(all_lines)
df["label"] = df.apply(lambda x: classify_lines(x,p90, p75), axis=1)

df.to_csv("lines_debug.csv", index=False)

In [16]:
df.head()

,text,size,is_bold,page,bbox,label
0,80 TH INTEGRATED ANNUAL,9.471,False,0,"(49.827701568603516, 89.02040100097656, 172.60...",body
1,REPORT 2024-25,11.000,False,0,"(49.82870101928711, 101.02137756347656, 125.52...",h2
2,"In a free enterprise, the community is not just",15.000,False,1,"(155.38999938964844, 607.6649780273438, 443.28...",h2
3,"another stakeholder in business, but is in fact",15.000,False,1,"(156.78599548339844, 625.6649780273438, 441.88...",h2
4,the very purpose of its existence.,15.000,False,1,"(194.72799682617188, 643.6649780273438, 400.54...",h2


## Observation: I think that first few pages are just noise for my dataset. Let's try removing it, and then again try making df again

In [17]:
import numpy as np

all_lines = []
for page_num,page in enumerate(doc):

    ## Let's remove page number < 2
    if page_num < 2:
        continue
    blocks = page.get_text("dict")["blocks"]
    all_lines.extend(build_lines(blocks,page_num))


sizes = [l["size"] for l in all_lines]

p90 = np.percentile(sizes,90)
p75 = np.percentile(sizes,75)
p50 = np.percentile(sizes,50)

In [18]:
i=0
for line in all_lines:
    if(i == 5):
        break

    if(is_noise(line["text"])):
        continue
    
    label = classify_lines(line, p90, p75)
    text = line["text"]
    print(f"The text is : {text}, and it's label is {label}, and all details are {line}")
    i = i+1

The text is : In Remembrance, and it's label is h2, and all details are {'text': 'In Remembrance', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (235.58900451660156, 86.29800415039062, 363.75262451171875, 111.28199768066406)}
The text is : 28.12.1937 – 09.10.2024, and it's label is h2, and all details are {'text': '28.12.1937 – 09.10.2024', 'size': 12.5, 'is_bold': False, 'page': 2, 'bbox': (240.5919952392578, 500.17498779296875, 354.6722412109375, 516.9000244140625)}
The text is : Padma Vibhushan, and it's label is h2, and all details are {'text': 'Padma Vibhushan', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (231.77000427246094, 455.2980041503906, 363.50653076171875, 480.2820129394531)}
The text is : Ratan N Tata, and it's label is h2, and all details are {'text': 'Ratan N Tata', 'size': 20.0, 'is_bold': False, 'page': 2, 'bbox': (245.86599731445312, 474.2200012207031, 349.4200134277344, 501.9800109863281)}
The text is : It is with a profound sense of loss that we bid far

In [19]:
import pandas as pd

df_2 = pd.DataFrame(all_lines)
df_2["label"] = df_2.apply(lambda x: classify_lines(x,p90, p75), axis=1)

df_2.to_csv("lines_debug.csv", index=False)

In [20]:
df_2.head(10)

,text,size,is_bold,page,bbox,label
0,In Remembrance,18.0,False,2,"(235.58900451660156, 86.29800415039062, 363.75...",h2
1,28.12.1937 – 09.10.2024,12.5,False,2,"(240.5919952392578, 500.17498779296875, 354.67...",h2
2,Padma Vibhushan,18.0,False,2,"(231.77000427246094, 455.2980041503906, 363.50...",h2
3,Ratan N Tata,20.0,False,2,"(245.86599731445312, 474.2200012207031, 349.42...",h2
4,It is with a profound sense of loss that we bi...,11.0,False,2,"(124.84500122070312, 540.7139892578125, 472.86...",h2
5,a truly uncommon leader whose immeasurable con...,11.0,False,2,"(125.14099884033203, 555.7139892578125, 472.61...",h2
6,only the Tata Group but also the very fabric o...,11.0,False,2,"(170.67300415039062, 570.7139892578125, 424.60...",h2
7,"For the Tata Group, Mr. Tata was more than a c...",11.0,False,2,"(136.0989990234375, 597.7139892578125, 461.639...",h2
8,example. With an unwavering commitment to exce...,11.0,False,2,"(141.36300659179688, 612.7139892578125, 456.38...",h2
9,"innovation, the Tata Group under his stewardsh...",11.0,False,2,"(123.62999725341797, 627.7139892578125, 474.13...",h2


## Let's see the labels, and try understanding about why it is happening like that

In [21]:
df_2["label"].value_counts()

label
body      35004
h2        23990
ignore     9279
h1          227
Name: count, dtype: int64

In [22]:
# for h1(most important)
df_2[df_2["label"] == 'h1']

,text,size,is_bold,page,bbox,label
39,Sales (wholesale excl. CJLR),11.0,True,4,"(51.02360153198242, 231.5299835205078, 157.664...",h1
40,units,9.5,True,4,"(51.02360153198242, 251.2530059814453, 69.8326...",h1
43,I crore,9.5,True,4,"(221.43299865722656, 251.2530059814453, 247.87...",h1
46,I crore,9.5,True,4,"(220.93899536132812, 437.25299072265625, 247.3...",h1
62,$ billion*,9.5,True,4,"(51.02360153198242, 437.25299072265625, 85.599...",h1
...,...,...,...,...,...,...
63537,"the Company for a period of five years, i.e. ,...",9.5,True,540,"(306.6344909667969, 323.2514953613281, 546.401...",h1
63558,Mr Chowdary has given his declaration to the B...,9.5,True,540,"(306.63446044921875, 593.2321166992188, 546.39...",h1
63559,"alia , confirming that (i) he meets the criter...",9.5,True,540,"(306.6344909667969, 605.2305908203125, 546.397...",h1
63749,"Regulations, 2015 (‘SEBI Listing Regulations’)...",9.5,True,542,"(306.63800048828125, 311.25494384765625, 546.3...",h1


## Observation: Boldness can't be a good feature for header, let's start with working on with word count + size factor

In [23]:
def word_count(text):
    return len(text.split())

def is_sentence(text):
    return word_count(text) > 8

def is_title_like(text):
    return text.isupper() or text.istitle()

In [24]:
def classify_lines(line, p90, p75):
    text = line["text"]
    wc = word_count(text)
    size = line['size']
    
    if is_noise(text):
        return "ignore"
    
    if wc > 8:
        return "body"
    
    ## Header
    if size >= p90 and wc<= 6:
        return "h1"
    
    # Sub header
    if size >= p75 and wc <= 8:
        return "h2"

    return "body"



In [25]:
import numpy as np

all_lines = []
for page_num,page in enumerate(doc):

    ## Let's remove page number < 2
    if page_num < 2:
        continue
    blocks = page.get_text("dict")["blocks"]
    all_lines.extend(build_lines(blocks,page_num))


sizes = [l["size"] for l in all_lines]

p90 = np.percentile(sizes,90)
p75 = np.percentile(sizes,75)
p50 = np.percentile(sizes,50)

i=0
for line in all_lines:
    if(i == 5):
        break

    if(is_noise(line["text"])):
        continue
    
    label = classify_lines(line, p90, p75)
    text = line["text"]
    print(f"The text is : {text}, and it's label is {label}, and all details are {line}")
    i = i+1

The text is : In Remembrance, and it's label is h1, and all details are {'text': 'In Remembrance', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (235.58900451660156, 86.29800415039062, 363.75262451171875, 111.28199768066406)}
The text is : 28.12.1937 – 09.10.2024, and it's label is h1, and all details are {'text': '28.12.1937 – 09.10.2024', 'size': 12.5, 'is_bold': False, 'page': 2, 'bbox': (240.5919952392578, 500.17498779296875, 354.6722412109375, 516.9000244140625)}
The text is : Padma Vibhushan, and it's label is h1, and all details are {'text': 'Padma Vibhushan', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (231.77000427246094, 455.2980041503906, 363.50653076171875, 480.2820129394531)}
The text is : Ratan N Tata, and it's label is h1, and all details are {'text': 'Ratan N Tata', 'size': 20.0, 'is_bold': False, 'page': 2, 'bbox': (245.86599731445312, 474.2200012207031, 349.4200134277344, 501.9800109863281)}
The text is : It is with a profound sense of loss that we bid far

In [27]:
import pandas as pd

df_3 = pd.DataFrame(all_lines)
df_3["label"] = df_3.apply(lambda x: classify_lines(x,p90, p75), axis=1)

df_3.to_csv("lines_debug.csv", index=False)

df_3.head(10)

,text,size,is_bold,page,bbox,label
0,In Remembrance,18.0,False,2,"(235.58900451660156, 86.29800415039062, 363.75...",h1
1,28.12.1937 – 09.10.2024,12.5,False,2,"(240.5919952392578, 500.17498779296875, 354.67...",h1
2,Padma Vibhushan,18.0,False,2,"(231.77000427246094, 455.2980041503906, 363.50...",h1
3,Ratan N Tata,20.0,False,2,"(245.86599731445312, 474.2200012207031, 349.42...",h1
4,It is with a profound sense of loss that we bi...,11.0,False,2,"(124.84500122070312, 540.7139892578125, 472.86...",body
5,a truly uncommon leader whose immeasurable con...,11.0,False,2,"(125.14099884033203, 555.7139892578125, 472.61...",body
6,only the Tata Group but also the very fabric o...,11.0,False,2,"(170.67300415039062, 570.7139892578125, 424.60...",body
7,"For the Tata Group, Mr. Tata was more than a c...",11.0,False,2,"(136.0989990234375, 597.7139892578125, 461.639...",body
8,example. With an unwavering commitment to exce...,11.0,False,2,"(141.36300659179688, 612.7139892578125, 456.38...",body
9,"innovation, the Tata Group under his stewardsh...",11.0,False,2,"(123.62999725341797, 627.7139892578125, 474.13...",body


In [ ]:
df_3[(df_3["label"] == 'h1') & (df_3["page"] == 255)] ## To analyze a single page

,text,size,is_bold,page,bbox,label
23230,Management Discussion and Analysis,21.0,False,255,"(51.02360153198242, 70.18099975585938, 375.620...",h1
23231,"Provident Fund Organization (“EPFO”), offered to",9.5,False,255,"(70.8666000366211, 110.25299835205078, 290.775...",h1
23236,fund exemption.,9.5,False,255,"(70.8666000366211, 170.24537658691406, 135.252...",h1
23275,agreed in the minutes.,9.5,False,255,"(70.8666000366211, 656.1987915039062, 158.5079...",h1
23276,Tax expenses / (credit):,10.0,False,255,"(70.86360168457031, 679.6099853515625, 167.722...",h1
23286,following reasons:,9.5,False,255,"(326.485595703125, 164.2409210205078, 397.2975...",h1
23290,"unabsorbed depreciation ₹763 crores, resulting in",9.5,False,255,"(346.3310852050781, 206.24037170410156, 546.40...",h1
23297,"During FY25, Jaguar Land Rover recognized",9.5,False,255,"(346.340576171875, 284.2352294921875, 546.4171...",h1
23300,previously unrecognized unused business losses.,9.5,False,255,"(346.340576171875, 320.2306823730469, 535.3931...",h1
23303,"of joint venture, joint operation, associates",9.5,False,255,"(346.3500671386719, 350.231689453125, 546.4246...",h1


### Observation: Since body size is dominating, and others are outlier, therefore percentile approach is failing 90 percentile is also too low. And it will be a common pattern of body size dominating in almost all financial reports. 

### Hypothesis 1: Instead of percentile, if we mind the mode size, we can get the body size, then we can use a threshold to add, and we can get our headings. 

In [37]:
from collections import Counter

size_counts = Counter(round(l['size'], 1) for l in all_lines)
body_size = size_counts.most_common(1)[0][0]

print("Body size:", body_size)

Body size: 8.5


In [38]:
## Can change these values
h2_threshold = body_size + 2.0
h1_threshold = body_size + 4.0

In [50]:
def classify_lines(line, body_size):
    text = line["text"]
    wc = len(text.split())
    size = line["size"]
    
    if is_noise(text):
        return "ignore"
    
    if wc > 8:
        return "body"
    
    if size >= body_size + 4 and wc <= 7:
        return 'h1'
    
    if size >= body_size + 2 and wc <= 9:
        return 'h2'
    
    return "body"



In [51]:
import numpy as np

all_lines = []
for page_num,page in enumerate(doc):

    ## Let's remove page number < 2
    if page_num < 2:
        continue
    blocks = page.get_text("dict")["blocks"]
    all_lines.extend(build_lines(blocks,page_num))

i=0
for line in all_lines:
    if(i == 5):
        break

    if(is_noise(line["text"])):
        continue
    
    label = classify_lines(line, body_size)
    text = line["text"]
    print(f"The text is : {text}, and it's label is {label}, and all details are {line}")
    i = i+1

The text is : In Remembrance, and it's label is h1, and all details are {'text': 'In Remembrance', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (235.58900451660156, 86.29800415039062, 363.75262451171875, 111.28199768066406)}
The text is : 28.12.1937 – 09.10.2024, and it's label is h1, and all details are {'text': '28.12.1937 – 09.10.2024', 'size': 12.5, 'is_bold': False, 'page': 2, 'bbox': (240.5919952392578, 500.17498779296875, 354.6722412109375, 516.9000244140625)}
The text is : Padma Vibhushan, and it's label is h1, and all details are {'text': 'Padma Vibhushan', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (231.77000427246094, 455.2980041503906, 363.50653076171875, 480.2820129394531)}
The text is : Ratan N Tata, and it's label is h1, and all details are {'text': 'Ratan N Tata', 'size': 20.0, 'is_bold': False, 'page': 2, 'bbox': (245.86599731445312, 474.2200012207031, 349.4200134277344, 501.9800109863281)}
The text is : It is with a profound sense of loss that we bid far

In [52]:
import pandas as pd

df_4 = pd.DataFrame(all_lines)
df_4["label"] = df_4.apply(lambda x: classify_lines(x,body_size), axis=1)

df_4.to_csv("lines_debug.csv", index=False)

df_4.head(10)

,text,size,is_bold,page,bbox,label
0,In Remembrance,18.0,False,2,"(235.58900451660156, 86.29800415039062, 363.75...",h1
1,28.12.1937 – 09.10.2024,12.5,False,2,"(240.5919952392578, 500.17498779296875, 354.67...",h1
2,Padma Vibhushan,18.0,False,2,"(231.77000427246094, 455.2980041503906, 363.50...",h1
3,Ratan N Tata,20.0,False,2,"(245.86599731445312, 474.2200012207031, 349.42...",h1
4,It is with a profound sense of loss that we bi...,11.0,False,2,"(124.84500122070312, 540.7139892578125, 472.86...",body
5,a truly uncommon leader whose immeasurable con...,11.0,False,2,"(125.14099884033203, 555.7139892578125, 472.61...",body
6,only the Tata Group but also the very fabric o...,11.0,False,2,"(170.67300415039062, 570.7139892578125, 424.60...",body
7,"For the Tata Group, Mr. Tata was more than a c...",11.0,False,2,"(136.0989990234375, 597.7139892578125, 461.639...",body
8,example. With an unwavering commitment to exce...,11.0,False,2,"(141.36300659179688, 612.7139892578125, 456.38...",body
9,"innovation, the Tata Group under his stewardsh...",11.0,False,2,"(123.62999725341797, 627.7139892578125, 474.13...",body


In [ ]:
df_4[(df_4["label"] == 'ignore') & (df_4["page"] == 317)]

text  size      is_bold  page  bbox                                                                           label 
(1)   6.260991  False    317   (338.7502746582031, 73.4930191040039, 348.11627197265625, 79.72576904296875)   ignore    1
-     6.260991  False    317   (372.749267578125, 465.9295959472656, 382.1152648925781, 467.6431884765625)    ignore    1
                               (365.6512756347656, 539.6312866210938, 375.01727294921875, 541.3449096679688)  ignore    1
                               (365.6512756347656, 574.5921020507812, 375.01727294921875, 576.3057250976562)  ignore    1
                               (365.6512756347656, 608.8585205078125, 375.01727294921875, 610.5721435546875)  ignore    1
                                                                                                                       ..
                               (294.21630859375, 438.54034423828125, 303.5823059082031, 440.2539367675781)    ignore    1
                             

## Observations: 

* H1 can be very big in word count. Word count should not be the limit like this one "Consolidated Statement of Changes in Equity for the year ended March 31, 2025
"
* H1 doesn't generally endswith .,;,,


In [61]:
def classify_lines(line, body_size):
    text = line["text"].strip()
    wc = len(text.split())
    size = line["size"]

    if is_noise(text):
        return "ignore"
    
    if text.endswith(('.',";",',')):
        return "body"
    
    # h1
    if size >= body_size + 4 :
        return "h1"
    
    if size >= body_size + 2 and wc <= 8:
        return "h2"
    
    if wc > 10:
        return "body"
    
    return "body"

In [62]:
import numpy as np

all_lines = []
for page_num,page in enumerate(doc):

    ## Let's remove page number < 2
    if page_num < 2:
        continue
    blocks = page.get_text("dict")["blocks"]
    all_lines.extend(build_lines(blocks,page_num))

i=0
for line in all_lines:
    if(i == 5):
        break

    if(is_noise(line["text"])):
        continue
    
    label = classify_lines(line, body_size)
    text = line["text"]
    print(f"The text is : {text}, and it's label is {label}, and all details are {line}")
    i = i+1

The text is : In Remembrance, and it's label is h1, and all details are {'text': 'In Remembrance', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (235.58900451660156, 86.29800415039062, 363.75262451171875, 111.28199768066406)}
The text is : 28.12.1937 – 09.10.2024, and it's label is h1, and all details are {'text': '28.12.1937 – 09.10.2024', 'size': 12.5, 'is_bold': False, 'page': 2, 'bbox': (240.5919952392578, 500.17498779296875, 354.6722412109375, 516.9000244140625)}
The text is : Padma Vibhushan, and it's label is h1, and all details are {'text': 'Padma Vibhushan', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (231.77000427246094, 455.2980041503906, 363.50653076171875, 480.2820129394531)}
The text is : Ratan N Tata, and it's label is h1, and all details are {'text': 'Ratan N Tata', 'size': 20.0, 'is_bold': False, 'page': 2, 'bbox': (245.86599731445312, 474.2200012207031, 349.4200134277344, 501.9800109863281)}
The text is : It is with a profound sense of loss that we bid far

In [63]:
import pandas as pd

df_5 = pd.DataFrame(all_lines)
df_5["label"] = df_5.apply(lambda x: classify_lines(x,body_size), axis=1)

df_5.to_csv("lines_debug.csv", index=False)

df_5.head(10)

,text,size,is_bold,page,bbox,label
0,In Remembrance,18.0,False,2,"(235.58900451660156, 86.29800415039062, 363.75...",h1
1,28.12.1937 – 09.10.2024,12.5,False,2,"(240.5919952392578, 500.17498779296875, 354.67...",h1
2,Padma Vibhushan,18.0,False,2,"(231.77000427246094, 455.2980041503906, 363.50...",h1
3,Ratan N Tata,20.0,False,2,"(245.86599731445312, 474.2200012207031, 349.42...",h1
4,It is with a profound sense of loss that we bi...,11.0,False,2,"(124.84500122070312, 540.7139892578125, 472.86...",body
5,a truly uncommon leader whose immeasurable con...,11.0,False,2,"(125.14099884033203, 555.7139892578125, 472.61...",body
6,only the Tata Group but also the very fabric o...,11.0,False,2,"(170.67300415039062, 570.7139892578125, 424.60...",body
7,"For the Tata Group, Mr. Tata was more than a c...",11.0,False,2,"(136.0989990234375, 597.7139892578125, 461.639...",body
8,example. With an unwavering commitment to exce...,11.0,False,2,"(141.36300659179688, 612.7139892578125, 456.38...",body
9,"innovation, the Tata Group under his stewardsh...",11.0,False,2,"(123.62999725341797, 627.7139892578125, 474.13...",body


In [68]:
df_5[(df_5["label"] == 'body') & (df_5["page"] == 254)] ## To analyze a single page

,text,size,is_bold,page,bbox,label
23075,CONSOLIDATED,8.0,False,254,"(334.8330078125, 788.9246215820312, 387.185028...",body
23076,FINANCIALS,8.0,False,254,"(334.8330078125, 798.5245971679688, 373.432952...",body
23078,STANDALONE,8.0,False,254,"(425.625, 788.9246215820312, 471.1097412109375...",body
23079,FINANCIALS,8.0,False,254,"(425.625, 798.5245971679688, 464.2249450683594...",body
23082,STATUTORY,8.0,False,254,"(243.04200744628906, 788.9371337890625, 281.87...",body
...,...,...,...,...,...,...
23220,(12),8.5,False,254,"(525.2754516601562, 683.14599609375, 539.04547...",body
23221,Total,8.5,False,254,"(330.48095703125, 696.4315185546875, 348.46359...",body
23224,FY25,9.5,False,254,"(326.4835205078125, 716.1295166015625, 345.811...",body
23225,Tata Motors Limited (the “Company”) in October...,9.5,False,254,"(326.4834899902344, 734.2554931640625, 546.395...",body


In [69]:
import pandas as pd

# mask for numeric values in 'text'
numeric_mask = pd.to_numeric(df_5['text'], errors='coerce').notna()

# mask for label == ignore
ignore_mask = df_5['label'] == 'ignore'

# count 1: both conditions
valid_count = df_5[numeric_mask & ignore_mask].shape[0]

# count 2: total ignore labels
total_ignore = df_5[ignore_mask].shape[0]

# percentage
percentage = (valid_count / total_ignore) * 100 if total_ignore > 0 else 0

print(valid_count, total_ignore, percentage)

7382 14986 49.2593086881089


## Observation: Around 50% of 'ignore' label is numerical, they definately have came from table

7382

## Now let's try making table LLM understandable

## The tables will be having numeric heavy lines


In [75]:
import re

def numeric_ratio(text):
    tokens = text.split()
    if not tokens:
        return 0
    
    numeric_tokens = sum(1 for t in tokens if re.search(r'\d',t))
    return numeric_tokens / len(tokens)

def is_table_line(text):
    return numeric_ratio(text) > 0.4


In [76]:
def classify_extended(line,body_size):
    base_label = classify_lines(line,body_size)
    text = line["text"]

    if base_label == "ignore":
        if is_table_line(text):
            return "table"
        return "ignore"
    
    if base_label == "body":
        if is_table_line(text):
            return "table"
        
    return base_label

In [77]:
import numpy as np

all_lines = []
for page_num,page in enumerate(doc):

    ## Let's remove page number < 2
    if page_num < 2:
        continue
    blocks = page.get_text("dict")["blocks"]
    all_lines.extend(build_lines(blocks,page_num))

i=0
for line in all_lines:
    if(i == 5):
        break

    if(is_noise(line["text"])):
        continue
    
    label = classify_extended(line, body_size)
    text = line["text"]
    print(f"The text is : {text}, and it's label is {label}, and all details are {line}")
    i = i+1

The text is : In Remembrance, and it's label is h1, and all details are {'text': 'In Remembrance', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (235.58900451660156, 86.29800415039062, 363.75262451171875, 111.28199768066406)}
The text is : 28.12.1937 – 09.10.2024, and it's label is h1, and all details are {'text': '28.12.1937 – 09.10.2024', 'size': 12.5, 'is_bold': False, 'page': 2, 'bbox': (240.5919952392578, 500.17498779296875, 354.6722412109375, 516.9000244140625)}
The text is : Padma Vibhushan, and it's label is h1, and all details are {'text': 'Padma Vibhushan', 'size': 18.0, 'is_bold': False, 'page': 2, 'bbox': (231.77000427246094, 455.2980041503906, 363.50653076171875, 480.2820129394531)}
The text is : Ratan N Tata, and it's label is h1, and all details are {'text': 'Ratan N Tata', 'size': 20.0, 'is_bold': False, 'page': 2, 'bbox': (245.86599731445312, 474.2200012207031, 349.4200134277344, 501.9800109863281)}
The text is : It is with a profound sense of loss that we bid far

In [78]:
import pandas as pd

df_6 = pd.DataFrame(all_lines)
df_6["label"] = df_6.apply(lambda x: classify_extended(x,body_size), axis=1)

df_6.to_csv("lines_debug.csv", index=False)

df_6.head(10)

,text,size,is_bold,page,bbox,label
0,In Remembrance,18.0,False,2,"(235.58900451660156, 86.29800415039062, 363.75...",h1
1,28.12.1937 – 09.10.2024,12.5,False,2,"(240.5919952392578, 500.17498779296875, 354.67...",h1
2,Padma Vibhushan,18.0,False,2,"(231.77000427246094, 455.2980041503906, 363.50...",h1
3,Ratan N Tata,20.0,False,2,"(245.86599731445312, 474.2200012207031, 349.42...",h1
4,It is with a profound sense of loss that we bi...,11.0,False,2,"(124.84500122070312, 540.7139892578125, 472.86...",body
5,a truly uncommon leader whose immeasurable con...,11.0,False,2,"(125.14099884033203, 555.7139892578125, 472.61...",body
6,only the Tata Group but also the very fabric o...,11.0,False,2,"(170.67300415039062, 570.7139892578125, 424.60...",body
7,"For the Tata Group, Mr. Tata was more than a c...",11.0,False,2,"(136.0989990234375, 597.7139892578125, 461.639...",body
8,example. With an unwavering commitment to exce...,11.0,False,2,"(141.36300659179688, 612.7139892578125, 456.38...",body
9,"innovation, the Tata Group under his stewardsh...",11.0,False,2,"(123.62999725341797, 627.7139892578125, 474.13...",body


In [81]:
df_6[(df_6["label"] == 'table') & (df_6["page"] == 231)] 

,text,size,is_bold,page,bbox,label
20358,2024-25,8.000,False,231,"(61.64540100097656, 799.6331176757812, 89.9349...",table
20359,228,7.500,False,231,"(519.97900390625, 793.3052368164062, 531.57476...",table
20364,18.5,8.500,False,231,"(418.1895751953125, 123.27899932861328, 433.61...",table
20366,2.4,8.500,False,231,"(420.4930725097656, 135.281005859375, 431.3212...",table
20368,11.5,8.500,False,231,"(418.3000793457031, 147.28302001953125, 433.51...",table
20370,6.1,8.500,False,231,"(420.5440673828125, 159.2850341796875, 431.272...",table
20372,16,8.500,False,231,"(421.6405944824219, 171.28704833984375, 430.17...",table
20374,1,8.500,False,231,"(423.7485656738281, 183.2890625, 428.058074951...",table
20376,55.5,8.500,False,231,"(418.30859375, 195.5715789794922, 433.50662231...",table
20400,9.,9.500,False,231,"(51.08998489379883, 422.1288757324219, 60.4883...",table


In [82]:
df_6[(df_6["label"] == 'body') & (df_6["page"] == 231)] 

,text,size,is_bold,page,bbox,label
20357,80 th Integrated Annual Report,7.65,False,231,"(61.64540100097656, 789.0324096679688, 164.685...",body
20361,Plant Locations,7.00,False,231,"(74.86609649658203, 112.72699737548828, 120.63...",body
20362,Total Roof Top PV Solar installed capacity til...,7.00,False,231,"(338.05908203125, 112.72699737548828, 513.7518...",body
20363,"Pimpri, Pune",8.50,False,231,"(74.86609649658203, 123.27899932861328, 119.47...",body
20365,"Chinchwad, Pune",8.50,False,231,"(74.8660888671875, 135.281005859375, 135.26879...",body
20367,Jamshedpur,8.50,False,231,"(74.8660888671875, 147.28302001953125, 117.395...",body
20369,Lucknow,8.50,False,231,"(74.8660888671875, 159.2850341796875, 105.9148...",body
20371,Pantnagar,8.50,False,231,"(74.8660888671875, 171.28704833984375, 110.691...",body
20373,Dharwad,8.50,False,231,"(74.8660888671875, 183.2890625, 106.8872833251...",body
20375,Total,8.50,False,231,"(74.8660888671875, 195.5715789794922, 92.06329...",body


## Observations: We got the table numbers on the 'table' label and row names on the 'body' label, now we will merge it using bbox's y value

In [83]:
all_lines.sort(key=lambda x: (x["page"], x["bbox"][1]))

In [85]:
## LEt's start by grouping lines by Y

def group_rows(lines, y_tol = 3):
    rows = []
    current_row = []

    for line in lines:
        if not current_row:
            current_row.append(line)
            continue

        prev_y = current_row[-1]["bbox"][1]
        curr_y = line["bbox"][1]

        if abs(curr_y - prev_y) < y_tol:
            current_row.append(line)
        else:
            rows.append(current_row)
            current_row = [line]
    
    if current_row:
        rows.append(current_row)

    return rows



In [86]:
## Detect table rows
def is_table_row(row):
    texts = [l["text"] for l in row]

    numeric_parts = sum(1 for t in texts if numeric_ratio(t) > 0.4)

    return numeric_parts >= 1


In [87]:
## Merging a row meaningfully
def row_to_text(row):
    texts = [l["text"] for l in row]

    if len(texts) == 1:
        return texts[0]
    
    return f"{texts[0]}: " + ", ".join(texts[1:])

In [91]:
## Now let's group rows in tables
def group_tables(rows):
    tables = []
    current_table = []

    for row in rows:
        if is_table_row(row):
            current_table.append(row)
        else:
            if current_table:
                tables.append(current_table)
                current_table = []
    
    if current_table:
        tables.append(current_table)
    
    return tables


In [94]:
def table_to_text(table):
    lines = []
    for row in table:
        lines.append(row_to_text(row))
    return "Table Data:\n" + "\n".join(lines)

In [97]:
## So let's implement
grouped_rows = group_rows(all_lines)
rtx_list = []

for row in grouped_rows:
    rtx = row_to_text(row)
    rtx_list.append(rtx)

## Group the tables
tables = group_tables(grouped_rows)

table_texts = [table_to_text(t) for t in tables]
table_texts[2000]

'Table Data:\nAs % of: consolidated, 2.09%, 3.02%, 0.32%, 1.33%, 0.00%, 1.97%, 19.12%, 0.00%, -0.23%, 0.01%, -0.13%, 0.34%, -0.07%, 0.00%, 0.18%, 0.00%, 0.01%, -0.01%, 0.00%, 0.00%, 0.14%, 0.56%, -0.03%, 0.00%, 0.11%, 11.51%, 0.00%, -0.04%, 0.00%, 8.42%, 72.37%, 0.00%, 39.95%, 0.98%, -0.03%, 0.00%, -0.04%, 0.00%, 0.70%, 0.17%, profit or loss\nAmount: 1,664, 5,742, 707, 8,640, 32,041, 141, 765, 4,971, -, 540, 163, (57), (31), 111, 60, -, (69), 41, 25, 319, 2,010, 549, 30, 8,542, (929), 48, 65,092, 7, 3, 65,132, 30,833, 24, 840, 1,172, 1, 78, 57, 1, 779, (₹ in crores), Net Assets, i.e. total assets'